# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is a Dataset object
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each `RecordSet`, `Field`, and `Column` is referenced by its `@id` according to the Croissant schema. Let's enumerate the record sets present in this dataset, and show their fields and columns via their `@id`s.

In [ ]:
# List all record sets and display their basic structure using @id fields
print("Available record sets and their fields (by '@id'):")
record_sets = []
for rset in metadata.record_sets:
    print(f"- RecordSet @id: {rset.id}  (name: {rset.name})")
    record_sets.append(rset.id)
    if rset.fields:
        for field in rset.fields:
            print(f"    - Field @id: {field.id}  (name: {field.name}, data_type: {field.data_type})")
    if rset.columns:
        for col in rset.columns:
            print(f"    - Column @id: {col.id}  (name: {col.name})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll attempt to load data for the primary `RecordSet` (often the first, but adapt as needed).

In [ ]:
# Extract data from each record set using their @id
# If no record sets are described, handle gracefully
dataframes = {}

if not record_sets:
    print("No record sets found in this schema, unable to load records.")
else:
    for record_set_id in record_sets:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")
            print(f"Columns (@id): {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"Error loading records from RecordSet @id: {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

To demonstrate, select a numeric field (by its `@id`) from the loaded DataFrame. We'll filter, normalize, and group as an example. Please replace the `numeric_field_id` and `group_field_id` variables with appropriate `@id` values from your record set if desired.

In [ ]:
# Example EDA on the first record set (modify variable values as appropriate)
if record_sets:
    first_rset = record_sets[0]
    df = dataframes[first_rset]
    print(f"Performing EDA on RecordSet @id: {first_rset}")
    
    # Guess numeric columns (if any)
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_cols:
        print("No numeric columns detected for EDA.")
    else:
        numeric_field_id = numeric_cols[0]  # Use first numeric field
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.90)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (90th percentile): {len(filtered_df)} selected.")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Guess a group field (first non-numeric string column)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped filtered data by {group_field_id}, showing mean of {numeric_field_id}:")
            display(grouped_df.head())
        else:
            print("No string field available to group.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll visualize the distribution of the selected numeric field and its normalized values, if available, using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and 'numeric_field_id' in locals():
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field_id} (Normalized, Filtered)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel("Count")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded metadata and records from the Mass Spectrometry Analysis dataset, referencing all data by their Croissant `@id`s for clarity and reproducibility.
- We explored the structure and identified available `RecordSet`s, `Field`s, and columns, then extracted records from the main record set for further analysis.
- Basic EDA operations were performed, including filtering, normalization, and grouping using example fields.
- Data distributions were visualized to reveal patterns in one of the numeric variables.

For further analysis, consult the Croissant schema and documentation to select fields of biological or scientific relevance to your task.